# Spark Example: Spotify Dataset Analysis
This example shows how to use Spark SQL to analyze a Spotify dataset. We will read the dataset, perform some queries, and display the results.   
Notice that there are several commands and techniques that have not been mentioned in the lecture. You may need to refer to the Spark documentation to understand how they work.

Note: this has been tested with pyspark 3.5.5.  If you are using newer version, you may encounter with type casting problems.

## Initilization
Define all parameters

In [1]:
host = 'local'
app_name = 'Spotify Analysis'
data_file_path = 'spotify.csv'

Perform all necessary initialization

In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import avg, col, max, desc, sum

In [3]:
spark = SparkSession.builder.master(host).appName(app_name).getOrCreate()
df = spark.read.option("header", True).csv(data_file_path)

25/11/23 17:07:39 WARN Utils: Your hostname, Natawuts-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 100.122.136.101 instead (on interface en0)
25/11/23 17:07:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 17:07:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/23 17:07:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Question 1: How many records are there? 
Provide answer in integer


In [4]:
print(df.count())

20718


## Question 2: Which track has the fastest tempo?  
Provide answer in string.

In [5]:
df = df.withColumn('Tempo', col('Tempo').cast('float'))
max_tempo = df.agg(max('Tempo')).first()[0]
v = df.filter(df['Tempo'] == max_tempo).select('Track').first()
print(v[0])


Call The Doctor


## Question 3: Given LVR is a ratio between likes and views, which artist has a track with the most LVR?
Provide answer in string.


First, we calculate LVR by dividing Likes and Views.  To ensure that we have no problem with datatypes, we first cast column types.

In [7]:
df = df.withColumn('Likes', col('Likes').cast('int'))
df = df.withColumn('Views', col('Views').cast('int'))
df = df.withColumn('LVR', col('Likes') / col('Views'))

In [8]:
max_lvr_df = df.agg(max('LVR').alias('max_lvr'))
max_lvr = max_lvr_df.first()[0]
v = df.filter(df['LVR'] == max_lvr).select('Artist').first()
print(v[0])

j-hope


## Question 4: Which artist has the most tracks in the dataset?  And how many?
Provide answer in tuple of (artist_name in string, track_count in integer).


In [9]:
most_tracks = df.groupby('Artist').count().orderBy(desc('count')).first()
answer = (most_tracks['Artist'], most_tracks['count'])
print(answer)

('Snoop Dogg', 10)


## Question 5: Which artist has the longest total duration of all tracks combined?
Provide answer in string.


In [10]:
df = df.withColumn('Duration_ms', col('Duration_ms').cast('float'))
total_duration = df.groupBy('Artist').agg(sum('Duration_ms').alias('artist_total_duration'))
max_duration = total_duration.agg(max('artist_total_duration')).first()[0]
v = total_duration.filter(total_duration['artist_total_duration'] == max_duration).select('Artist').first()
print(v[0])

Ankit Tiwari


In [11]:
spark.stop()